# Palm Vein Metric Learning — Colab
**ResNet-18 + Triplet Loss | 4 train / 1 test per subject**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, Sampler
from PIL import Image
import numpy as np, os, random, time, matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.model_selection import KFold
torch.manual_seed(42); random.seed(42); np.random.seed(42)
print('Imports OK')

In [ ]:
DATASET           = '/content/drive/MyDrive/palmvein/palm_vein_ready'
EMBEDDING_DIM     = 128
DROPOUT           = 0.5
P                 = 15
K                 = 4
BATCHES_PER_EPOCH = 20
EPOCHS            = 50
LR                = 1e-4
MARGIN            = 0.2
N_FOLDS           = 5
IMAGENET_MEAN     = [0.485, 0.456, 0.406]
IMAGENET_STD      = [0.229, 0.224, 0.225]
all_subjects = sorted([d for d in os.listdir(DATASET)
                        if os.path.isdir(os.path.join(DATASET, d))])
print(f'Subjects: {len(all_subjects)} | Batch size: {P*K}')

In [ ]:
train_transform = T.Compose([
    T.Grayscale(num_output_channels=3),
    T.RandomRotation(degrees=180, fill=0),
    T.RandomResizedCrop(224, scale=(0.85,1.15), ratio=(0.95,1.05)),
    T.ColorJitter(brightness=0.1),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
val_transform = T.Compose([
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
print('Transforms ready.')

In [ ]:
class PalmVeinDataset(Dataset):
    def __init__(self, root, subjects, sample_indices, transform=None):
        self.transform = transform
        self.samples   = []
        self.label_to_indices = defaultdict(list)
        for label, subj in enumerate(subjects):
            folder = os.path.join(root, subj)
            files  = sorted([f for f in os.listdir(folder) if f.endswith('.png')])
            for si in sample_indices:
                if si < len(files):
                    idx = len(self.samples)
                    self.samples.append((os.path.join(folder, files[si]), label))
                    self.label_to_indices[label].append(idx)
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path)
        if self.transform: img = self.transform(img)
        return img, label

train_ds = PalmVeinDataset(DATASET, all_subjects, [0,1,2,3], transform=train_transform)
val_ds   = PalmVeinDataset(DATASET, all_subjects, [4],       transform=val_transform)
print(f'Train: {len(train_ds)} imgs | Test: {len(val_ds)} imgs')

In [ ]:
class PKSampler(Sampler):
    def __init__(self, l2i, P, K, B):
        self.l2i=l2i; self.P=P; self.K=K; self.B=B
        self.labels=list(l2i.keys())
    def __iter__(self):
        for _ in range(self.B):
            chosen = random.sample(self.labels, self.P)
            yield from [i for c in chosen for i in random.choices(self.l2i[c], k=self.K)]
    def __len__(self): return self.B*self.P*self.K

pk_sampler   = PKSampler(train_ds.label_to_indices, P, K, BATCHES_PER_EPOCH)
train_loader = DataLoader(train_ds, batch_size=P*K, sampler=pk_sampler)
print('PK Sampler ready.')

In [ ]:
class EmbeddingNet(nn.Module):
    def __init__(self, embedding_dim=EMBEDDING_DIM, dropout=DROPOUT):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        for p in backbone.parameters():        p.requires_grad = False
        for p in backbone.layer4.parameters(): p.requires_grad = True
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        self.head = nn.Sequential(
            nn.Flatten(), nn.Dropout(p=dropout), nn.Linear(512, embedding_dim))
    def forward(self, x):
        return F.normalize(self.head(self.backbone(x)), p=2, dim=1)
    def train(self, mode=True):
        super().train(mode)
        for m in self.modules():
            if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)): m.eval()
        return self

model     = EmbeddingNet().to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)')
print('Output shape:', model(torch.zeros(1,3,224,224).to(device)).shape)

In [ ]:
def batch_hard_triplet_loss(embeddings, labels, margin=MARGIN):
    dist     = torch.cdist(embeddings, embeddings, p=2).pow(2)
    lc       = labels.unsqueeze(1)
    pos_mask = (lc == lc.T);   pos_mask.fill_diagonal_(False)
    neg_mask = (lc != lc.T)
    hp = (dist * pos_mask.float()).max(dim=1).values
    dn = dist.clone(); dn[~neg_mask] = float('inf')
    hn = dn.min(dim=1).values
    return F.relu(hp - hn + margin).mean()

print('Triplet loss ready.')

In [ ]:
def train_one_epoch(model, loader, optimiser):
    model.train()
    total, n = 0.0, 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimiser.zero_grad()
        loss = batch_hard_triplet_loss(model(imgs), lbls)
        loss.backward(); optimiser.step()
        total += loss.item(); n += 1
    return total / max(n, 1)


def evaluate_nn(model, gallery_ds, query_ds):
    """Gallery = train embeddings. Query = test embeddings. Returns NN accuracy."""
    model.eval()
    def embed(ds):
        embs, lbls = [], []
        with torch.no_grad():
            for imgs, lbl in DataLoader(ds, batch_size=32):
                embs.append(model(imgs.to(device)).cpu())
                lbls.extend(lbl.tolist())
        return torch.cat(embs), torch.tensor(lbls)
    g_emb, g_lbl = embed(gallery_ds)
    q_emb, q_lbl = embed(query_ds)
    dist  = torch.cdist(q_emb, g_emb, p=2)
    preds = g_lbl[dist.argmin(dim=1)]
    return (preds == q_lbl).float().mean().item()

print('Functions ready.')

In [ ]:
model     = EmbeddingNet().to(device)
optimiser = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

train_losses, val_accs = [], []
print(f'Training {EPOCHS} epochs on {device}...')
print('-' * 55)

for epoch in range(1, EPOCHS + 1):
    t0   = time.time()
    loss = train_one_epoch(model, train_loader, optimiser)
    acc  = evaluate_nn(model, train_ds, val_ds)
    train_losses.append(loss)
    val_accs.append(acc)
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS}  loss={loss:.4f}  '
              f'val_acc={acc*100:.1f}%  ({time.time()-t0:.1f}s)')

print('-' * 55)
best = val_accs.index(max(val_accs)) + 1
print(f'Best: {max(val_accs)*100:.1f}% at epoch {best}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13,4))
ax1.plot(train_losses, color='steelblue')
ax1.set_title('Triplet Loss'); ax1.set_xlabel('Epoch'); ax1.grid(True, alpha=0.3)
ax2.plot([a*100 for a in val_accs], color='darkorange')
ax2.axhline(max(val_accs)*100, color='red', ls='--', alpha=0.6,
            label=f'Best: {max(val_accs)*100:.1f}%')
ax2.set_title('NN Identification Accuracy'); ax2.set_xlabel('Epoch')
ax2.set_ylim(0, 105); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/palmvein/training_curves.png', dpi=150)
plt.show()

## 5-Fold Cross-Validation
Skip if you just want the single-run results.

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
subjects_arr = np.array(all_subjects)
fold_accs = []

for fold, (tr_idx, vl_idx) in enumerate(kf.split(subjects_arr), 1):
    tr_s = subjects_arr[tr_idx].tolist()
    vl_s = subjects_arr[vl_idx].tolist()
    tr_f  = PalmVeinDataset(DATASET, tr_s, [0,1,2,3], transform=train_transform)
    vl_f  = PalmVeinDataset(DATASET, vl_s, [4],       transform=val_transform)
    vl_g  = PalmVeinDataset(DATASET, vl_s, [0,1,2,3], transform=val_transform)
    sampler_f = PKSampler(tr_f.label_to_indices, min(P,len(tr_s)), K, BATCHES_PER_EPOCH)
    loader_f  = DataLoader(tr_f, batch_size=P*K, sampler=sampler_f)
    m   = EmbeddingNet().to(device)
    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, m.parameters()), lr=LR)
    for _ in range(EPOCHS): train_one_epoch(m, loader_f, opt)
    acc = evaluate_nn(m, vl_g, vl_f)
    fold_accs.append(acc)
    print(f'Fold {fold}/{N_FOLDS}  acc={acc*100:.1f}%')

print(f'Mean: {np.mean(fold_accs)*100:.1f}%  Std: {np.std(fold_accs)*100:.1f}%')